# 04 - Coop Navigation: from MADDPG to MAPPO, coordination and scaling

*N* agents must cover *N* landmarks without colliding. Fully cooperative.

1. **MADDPG vs MAPPO** - why do we switch algorithm family for cooperative teams?
2. **IPPO vs MAPPO** - does the centralised critic help?
3. **Parameter sharing** - one network for everybody, or one each? And what happens with more agents?

*(Pre-trained runs.)*

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from marl.analysis import (load_run, load_policy, evaluate, video, evolution_frames, plot_curves,
                           plot_matrix, moving_target_matrix, crossplay_matrix, speaker_messages,
                           message_separation, message_drift, pick)
from marl.render import show_video
plt.rcParams["figure.dpi"] = 90

## 1. MADDPG vs MAPPO
MADDPG works here too. The question is what it costs: its critic takes **every agent's observation and action** - with N agents that input grows with N,
and learning is off-policy from a replay buffer. MAPPO's critic only estimates the value of the global state, and on-policy PPO combines naturally with many parallel
environments and shared networks (next notebooks: vectorised simulators).

In [ ]:
maddpg3, mappo3 = load_run("spread_maddpg_N3"), load_run("spread_mappo_unshared_N3")
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
plot_curves([maddpg3, mappo3], "landmark_dist", ax=ax[0], title="avg distance landmark -> nearest agent")
plot_curves([maddpg3, mappo3], "landmark_dist", ax=ax[1], x="wall_time", title="same, against wall-clock")
plt.tight_layout()

In [ ]:
# Cost of training, measured on the runs above (all trained side by side on the same laptop).
for label, name in [("MADDPG", "spread_maddpg_N3"), ("MAPPO", "spread_mappo_unshared_N3"),
                    ("MADDPG", "spread_maddpg_N6"), ("MAPPO", "spread_mappo_unshared_N6")]:
    r = load_run(name)
    n = r["config"]["env_kwargs"]["N"]
    minutes_per_M = r["log"]["wall_time"][-1] / 60 / (r["log"]["step"][-1] / 1e6)
    obs = 4 + 2 * n + 2 * (n - 1) + 2 * (n - 1)            # vel, pos, landmarks, other agents, comm
    critic_in = n * obs + (n * 5 if label == "MADDPG" else 0)
    print(f"{label:<7} N={n}: {minutes_per_M:5.1f} min per 1M env steps | {n} critics, each sees {critic_in} numbers")

## 2. IPPO vs MAPPO
Same networks, same everything - only the critic's input differs (own observation vs global state).

In [ ]:
ippo3 = load_run("spread_ippo_N3")
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
plot_curves([mappo3, ippo3], "landmark_dist", ax=ax[0], title="avg landmark distance")
plot_curves([mappo3, ippo3], "coverage", ax=ax[1], title="fraction of landmarks covered")
plt.tight_layout()

On this task the two end up **equally good** (we checked three random seeds: sometimes IPPO is even faster).
A centralised critic is not automatically a win. Compare notebook 02 (Speaker-Listener): there the other agent's changing behaviour *is* the
difficulty, and the centralised critic made a large difference. Here each agent mostly needs to find a free landmark.

In [ ]:
show_video(evolution_frames(mappo3, k=4, seed=3), fps=10)

## 3. Parameter sharing: one brain for everybody?
Sharing one network across agents means fewer parameters and every agent's experience trains the same network - usually a big win.
But here the shared agents are **interchangeable**: from the same kind of view they all make the same kind of decision, so nobody commits to a landmark -
they **huddle in the middle**. Separate networks let each agent specialise.

In [ ]:
shared3 = load_run("spread_mappo_N3")
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
plot_curves([mappo3, shared3], "landmark_dist", ax=ax[0], title="avg landmark distance")
plot_curves([mappo3, shared3], "coverage", ax=ax[1], title="landmarks covered")
plt.tight_layout()
show_video(video(load_policy(shared3["ckpts"][-1]), "spread", {"N": 3}, title="shared MAPPO (final)") +
           video(load_policy(mappo3["ckpts"][-1]), "spread", {"N": 3}, title="unshared MAPPO (final)"))

### More agents: N = 3 -> 4 -> 6
Same algorithm, same settings, same training budget per run - only the team size changes.

In [ ]:
names = ["spread_mappo_unshared_N3", "spread_mappo_unshared_N4", "spread_mappo_unshared_N6",
         "spread_mappo_N3", "spread_mappo_N4", "spread_mappo_N6", "spread_maddpg_N6"]
runs = [load_run(n) for n in names]
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
plot_curves(runs[:3], "landmark_dist", ax=ax[0], title="MAPPO (separate networks)")
plot_curves(runs[3:], "landmark_dist", ax=ax[1], title="MAPPO (shared network) + MADDPG N=6")
plt.tight_layout()

With 3 agents the team learns to cover all landmarks in about half a million steps. With **4 or 6 agents our runs barely improve on
random behaviour**, even with several times more training. That cliff *is* the scalability problem of MARL:

* every agent gets the same team reward, so it gets harder to tell **which agent's** action helped (credit assignment - next section of the workshop);
* the agents must jointly discover a much larger set of "who goes where" assignments (exploration);
* in a crowded arena some landmarks sit closer together than two agents can, so collisions become unavoidable.

What helps: far more experience per second (vectorised simulators, coming next), better credit assignment (value decomposition), and curricula (start small, grow the team).

In [ ]:
show_video(video(load_policy(runs[2]["ckpts"][-1]), "spread", {"N": 6}, seed=3, title="MAPPO N=6 (final)"))